In [4]:
from collections import Counter

import torch
from tqdm.auto import tqdm
tqdm.pandas()

import altair as alt
import os
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import numpy as np
from scipy.stats import pearsonr

import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from natsort import natsorted
import logomaker

from utils import add_germline_information, add_column_aa_one_mutation_away_from_codon

from netam import framework
from dnsmex import dxsm_data, dnsm_zoo, dasm_zoo
from dnsmex.dasm_oe import write_sites_oe, OEPlotter
from dnsmex.local import localify
import tqdm

from netam.common import heavy_chain_shim
from netam.framework import load_crepe, load_pcp_df
from netam.oe_plot import annotate_sites_df
from netam.sequences import translate_sequence, AA_STR_SORTED
from netam.codon_table import single_mutant_aa_indices

from dnsmex.dasm_viz import dms_style_heatmap
from dnsmex.local import localify
from dnsmex.dxsm_data import pcp_df_of_nickname

figures_dir = localify("FIGURES_DIR")

In [3]:
load_pcp_df('/fh/fast/matsen_e/shared/bcr-mut-sel/pcps/v3/anarci/rodriguez-airr-seq-race-prod-NoWinCheck_igh_chothia.csv')

AssertionError: No heavy or light chain columns found in PCP file! Found columns Index(['Id', 'domain_no', 'hmm_species', 'chain_type', 'e-value', 'score',
       'seqstart_index', 'seqend_index', 'identity_species', 'v_gene',
       ...
       '104.2', '105', '106', '107', '108', '109', '110', '111', '112', '113'],
      dtype='object', length=163)

In [5]:
import pandas as pd
import numpy as np

# Read both files
file1 = "/fh/fast/matsen_e/shared/bcr-mut-sel/pcps/v3/anarci/rodriguez-airr-seq-race-prod-NoWinCheck_igh_chothia.csv"
file2 = "/fh/fast/matsen_e/shared/bcr-mut-sel/pcps/v3/anarci/rodriguez-airr-seq-race-prod-NoWinCheck_igh_imgt.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Get the position columns (everything after the metadata columns)
metadata_cols = ['Id', 'domain_no', 'hmm_species', 'chain_type', 'e-value', 'score', 
                 'seqstart_index', 'seqend_index', 'identity_species', 'v_gene', 
                 'v_identity', 'j_gene', 'j_identity']

# Get position column names for each scheme
positions1 = [col for col in df1.columns if col not in metadata_cols]
positions2 = [col for col in df2.columns if col not in metadata_cols]

print(f"Numbering scheme 1 (File 1) has {len(positions1)} positions")
print(f"Numbering scheme 2 (File 2) has {len(positions2)} positions")

# Create mapping for each sequence
mappings = []

# Get common sequence IDs
common_ids = set(df1['Id']) & set(df2['Id'])
print(f"\nFound {len(common_ids)} common sequences")

for seq_id in list(common_ids)[:10]:  # Test with first 10 sequences
    # Get the sequence from both files
    seq1_row = df1[df1['Id'] == seq_id].iloc[0]
    seq2_row = df2[df2['Id'] == seq_id].iloc[0]
    
    # Extract amino acids for each position
    seq1_aa = [seq1_row[pos] for pos in positions1]
    seq2_aa = [seq2_row[pos] for pos in positions2]
    
    # Create position mapping by matching amino acids
    pos_map = []
    j = 0  # pointer for seq2
    
    for i, aa1 in enumerate(seq1_aa):
        if aa1 != '-' and pd.notna(aa1):
            # Find matching amino acid in seq2
            while j < len(seq2_aa):
                aa2 = seq2_aa[j]
                if aa2 == aa1:
                    pos_map.append({
                        'sequence_id': seq_id,
                        'scheme1_position': positions1[i],
                        'scheme2_position': positions2[j],
                        'amino_acid': aa1,
                        'scheme1_index': i,
                        'scheme2_index': j
                    })
                    j += 1
                    break
                elif aa2 != '-' and pd.notna(aa2):
                    # Mismatch - might indicate insertion/deletion
                    print(f"Warning: Mismatch at {seq_id}: pos1={positions1[i]} ({aa1}) vs pos2={positions2[j]} ({aa2})")
                    j += 1
                    break
                else:
                    j += 1
    
    mappings.extend(pos_map)

# Convert to DataFrame
mapping_df = pd.DataFrame(mappings)

# Create a general position mapping (most common mapping across all sequences)
position_mapping = mapping_df.groupby(['scheme1_position', 'scheme2_position']).size().reset_index(name='count')
position_mapping = position_mapping.sort_values('count', ascending=False)

print("\nPosition mapping between schemes:")
print(position_mapping.head(20))

# Save the mapping
mapping_df.to_csv('sequence_position_mappings.csv', index=False)
position_mapping.to_csv('general_position_mapping.csv', index=False)

print("\nSaved detailed mappings to 'sequence_position_mappings.csv'")
print("Saved general position mapping to 'general_position_mapping.csv'")

Numbering scheme 1 (File 1) has 150 positions
Numbering scheme 2 (File 2) has 153 positions

Found 31738 common sequences

Position mapping between schemes:
   scheme1_position scheme2_position  count
0                 1                1     10
1                10               11     10
66               28               29     10
63               25               26     10
59               21               22     10
62               24               25     10
61               23               24     10
60               22               23     10
55               18               19     10
54               17               18     10
53               16               17     10
52               15               16     10
51               14               15     10
58               20               21     10
57                2                2     10
56               19               20     10
48              113              128     10
49               12               13     10
50     

In [ ]:
## TODO: take the VJ gene annotations from the pcp (which would be from partis and not from anarci and is better). Then check if they are consistent across v genes. Hopefully htye are, and then we can take the mapping from there.


